<a href="https://colab.research.google.com/github/JaimeTS/hyrox-analysis/blob/main/notebooks/01_scraping_hyrox_limpio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Scraping HYROX Barcelona 2026

1. Librerías y configuración
2. Funciones generales
   2.1 extraer_ranking_pagina()
   2.2 descargar_categoria_completa()
   2.3 extraer_splits_atleta()
   2.4 descargar_splits_categoria()
3. Parámetros del evento
4. HYROX Men Barcelona 2026
5. HYROX Women Barcelona 2026
6. Unión Men + Women

## 1. Librerías y configuración

In [ ]:
import os
import math
import time
from io import StringIO

import requests
import pandas as pd
from bs4 import BeautifulSoup

BASE_URL = "https://www.hyresult.com"

headers = {
    "User-Agent": "Mozilla/5.0"
}

RUTA_DATOS = "/content/drive/MyDrive/Colab Notebooks/Hyrox"

os.makedirs(RUTA_DATOS, exist_ok=True)

## 2. Funciones generales

### 2.1 Función para extraer una página del ranking

In [ ]:
def extraer_ranking_pagina(url):
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    rows = soup.find_all("tr")
    resultados = []

    i = 0

    while i < len(rows):
        cells = rows[i].find_all("td")
        textos = [c.get_text(" ", strip=True) for c in cells]

        # ==========================================================
        # CASO NORMAL
        # ==========================================================
        if len(cells) >= 7:

            posicion = cells[1].get_text(strip=True)
            posicion_ag = cells[2].get_text(strip=True)

            name_cell = cells[3]
            link_athlete = name_cell.find("a")
            img_flag = name_cell.find("img")

            nombre = link_athlete.get_text(strip=True) if link_athlete else None
            athlete_url = (
                link_athlete["href"]
                if link_athlete and link_athlete.has_attr("href")
                else None
            )

            pais = (
                img_flag["title"]
                if img_flag and img_flag.has_attr("title")
                else None
            )

            grupo_edad = cells[4].get_text(strip=True)
            tiempo = cells[5].get_text(strip=True)

            analyze_link = cells[6].find("a")
            analyze_url = (
                analyze_link["href"]
                if analyze_link and analyze_link.has_attr("href")
                else None
            )

            if nombre and tiempo and analyze_url:

                resultados.append({
                    "posicion": posicion,
                    "posicion_grupo_edad": posicion_ag,
                    "nombre": nombre,
                    "pais": pais,
                    "grupo_edad": grupo_edad,
                    "tiempo": tiempo,
                    "athlete_url": athlete_url,
                    "analyze_url": analyze_url
                })

            i += 1
            continue

        # ==========================================================
        # CASO ESPECIAL A
        # ==========================================================
        if len(cells) >= 3 and i + 4 < len(rows):

            posicion = cells[1].get_text(strip=True)
            posicion_ag = cells[2].get_text(strip=True)

            row_nombre = rows[i + 1]
            row_grupo = rows[i + 2]
            row_tiempo = rows[i + 3]
            row_analyze = rows[i + 4]

            nombre_cell = row_nombre.find("td")
            link_athlete = nombre_cell.find("a") if nombre_cell else None
            img_flag = nombre_cell.find("img") if nombre_cell else None

            nombre = link_athlete.get_text(strip=True) if link_athlete else None
            athlete_url = (
                link_athlete["href"]
                if link_athlete and link_athlete.has_attr("href")
                else None
            )

            pais = (
                img_flag["title"]
                if img_flag and img_flag.has_attr("title")
                else None
            )

            grupo_edad = row_grupo.get_text(" ", strip=True)
            tiempo = row_tiempo.get_text(" ", strip=True)

            analyze_link = row_analyze.find("a")
            analyze_url = (
                analyze_link["href"]
                if analyze_link and analyze_link.has_attr("href")
                else None
            )

            if nombre and tiempo and analyze_url:

                resultados.append({
                    "posicion": posicion,
                    "posicion_grupo_edad": posicion_ag,
                    "nombre": nombre,
                    "pais": pais,
                    "grupo_edad": grupo_edad,
                    "tiempo": tiempo,
                    "athlete_url": athlete_url,
                    "analyze_url": analyze_url
                })

                i += 5
                continue

        # ==========================================================
        # CASO ESPECIAL C
        # (fila vacía + datos repartidos)
        # ==========================================================
        if len(cells) == 1 and textos == [""] and i + 6 < len(rows):

            row_pos = rows[i + 1]
            row_pos_ag = rows[i + 2]
            row_nombre = rows[i + 3]
            row_grupo = rows[i + 4]
            row_tiempo = rows[i + 5]
            row_analyze = rows[i + 6]

            posicion = row_pos.get_text(" ", strip=True)
            posicion_ag = row_pos_ag.get_text(" ", strip=True)
            nombre = row_nombre.get_text(" ", strip=True)
            grupo_edad = row_grupo.get_text(" ", strip=True)
            tiempo = row_tiempo.get_text(" ", strip=True)

            analyze_link = row_analyze.find("a")
            analyze_url = (
                analyze_link["href"]
                if analyze_link and analyze_link.has_attr("href")
                else None
            )

            if (
                posicion.isdigit()
                and posicion_ag.isdigit()
                and nombre
                and tiempo
                and analyze_url
            ):

                resultados.append({
                    "posicion": posicion,
                    "posicion_grupo_edad": posicion_ag,
                    "nombre": nombre,
                    "pais": None,
                    "grupo_edad": grupo_edad,
                    "tiempo": tiempo,
                    "athlete_url": None,
                    "analyze_url": analyze_url
                })

                i += 7
                continue

        # ==========================================================
        # CASO ESPECIAL B
        # ==========================================================
        if len(cells) >= 2 and i + 5 < len(rows):

            posicion = cells[1].get_text(strip=True)

            row_pos_ag = rows[i + 1]
            row_nombre = rows[i + 2]
            row_grupo = rows[i + 3]
            row_tiempo = rows[i + 4]
            row_analyze = rows[i + 5]

            posicion_ag = row_pos_ag.get_text(" ", strip=True)

            nombre_cell = row_nombre.find("td")
            link_athlete = nombre_cell.find("a") if nombre_cell else None
            img_flag = nombre_cell.find("img") if nombre_cell else None

            nombre = link_athlete.get_text(strip=True) if link_athlete else None
            athlete_url = (
                link_athlete["href"]
                if link_athlete and link_athlete.has_attr("href")
                else None
            )

            pais = (
                img_flag["title"]
                if img_flag and img_flag.has_attr("title")
                else None
            )

            grupo_edad = row_grupo.get_text(" ", strip=True)
            tiempo = row_tiempo.get_text(" ", strip=True)

            analyze_link = row_analyze.find("a")
            analyze_url = (
                analyze_link["href"]
                if analyze_link and analyze_link.has_attr("href")
                else None
            )

            if nombre and tiempo and analyze_url:

                resultados.append({
                    "posicion": posicion,
                    "posicion_grupo_edad": posicion_ag,
                    "nombre": nombre,
                    "pais": pais,
                    "grupo_edad": grupo_edad,
                    "tiempo": tiempo,
                    "athlete_url": athlete_url,
                    "analyze_url": analyze_url
                })

                i += 6
                continue

        i += 1

    df = pd.DataFrame(resultados)

    if len(df) > 0:

        df["athlete_url"] = df["athlete_url"].apply(
            lambda x: BASE_URL + x if isinstance(x, str) and x.startswith("/") else x
        )

        df["analyze_url"] = df["analyze_url"].apply(
            lambda x: BASE_URL + x if isinstance(x, str) and x.startswith("/") else x
        )

    return df

### 2.2 Función para descargar una categoría completa

In [ ]:
def descargar_categoria_completa(ranking_url, total_atletas, pausa=0.5):
    participantes_por_pagina = 100
    num_paginas = math.ceil(total_atletas / participantes_por_pagina)

    dfs = []

    for pagina in range(1, num_paginas + 1):
        url = f"{ranking_url}?p={pagina}"

        print(f"Descargando página {pagina}/{num_paginas}")

        df_pagina = extraer_ranking_pagina(url)
        dfs.append(df_pagina)

        time.sleep(pausa)

    df_categoria = pd.concat(dfs, ignore_index=True)

    return df_categoria

### 2.3 Función robusta para extraer splits

In [ ]:
def extraer_splits_atleta(fila_atleta):
    url_resultado = fila_atleta["analyze_url"]
    url_splits = url_resultado + "?tab=splits"

    response = requests.get(url_splits, headers=headers)

    tablas = pd.read_html(StringIO(response.text))

    tablas_limpias = []

    for tabla in tablas:
        if tabla.shape[1] != 4:
            continue

        tabla = tabla.copy()
        tabla.columns = ["Split", "Time of Day", "Time", "Diff"]

        tabla = tabla.dropna(how="all")
        tabla = tabla[tabla["Split"].astype(str) != "Split"]

        tablas_limpias.append(tabla)

    if len(tablas_limpias) == 0:
        raise ValueError("No se encontraron tablas válidas de splits")

    df_splits = pd.concat(tablas_limpias, ignore_index=True)

    df_splits["posicion"] = fila_atleta["posicion"]
    df_splits["nombre"] = fila_atleta["nombre"]
    df_splits["grupo_edad"] = fila_atleta["grupo_edad"]
    df_splits["pais"] = fila_atleta["pais"]
    df_splits["tiempo_total"] = fila_atleta["tiempo"]
    df_splits["analyze_url"] = fila_atleta["analyze_url"]

    return df_splits